1. Projektni Zadatak: Razvoj Sistema Preporuke u
PySpark-u

1. Najpopularniji filmovi

In [60]:
# Mount GDrive & Import dataset
#from google.colab import drive
#drive.mount("/content/gdrive")

#dataset_path = "/content/gdrive/MyDrive/ml-100k"

from pathlib import Path

# Use workspace-local dataset path so the notebook runs on this project directly.
dataset_path = str((Path.cwd() / "ml-100k").resolve())
print(dataset_path)

/Users/vesco/Documents/Projects/big-data-ml/ml-100k


In [61]:
# Conenct to Spark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("MovieLensAnalysis").getOrCreate()

In [62]:
# Import ratings
ratings = spark.read.csv(f"{dataset_path}/u.data", sep="\t", inferSchema=True, header=False)
ratings = ratings.toDF("user_id", "movie_id", "rating", "timestamp")
ratings.show(5)

+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|    196|     242|     3|881250949|
|    186|     302|     3|891717742|
|     22|     377|     1|878887116|
|    244|      51|     2|880606923|
|    166|     346|     1|886397596|
+-------+--------+------+---------+
only showing top 5 rows


In [63]:
# Import movies
movies = spark.read.csv(f"{dataset_path}/u.item", sep="|", inferSchema=True, header=False)
movies = movies.selectExpr("_c0 as movie_id", "_c1 as title")
movies.show(5, truncate=False)

+--------+-----------------+
|movie_id|title            |
+--------+-----------------+
|1       |Toy Story (1995) |
|2       |GoldenEye (1995) |
|3       |Four Rooms (1995)|
|4       |Get Shorty (1995)|
|5       |Copycat (1995)   |
+--------+-----------------+
only showing top 5 rows


In [64]:
# Import users
users = spark.read.csv(f"{dataset_path}/u.user", sep="|", inferSchema=True, header=False)
users = users.toDF("user_id", "age", "gender", "occupation", "zip_code")
users.show(5)

+-------+---+------+----------+--------+
|user_id|age|gender|occupation|zip_code|
+-------+---+------+----------+--------+
|      1| 24|     M|technician|   85711|
|      2| 53|     F|     other|   94043|
|      3| 23|     M|    writer|   32067|
|      4| 24|     M|technician|   43537|
|      5| 33|     F|     other|   15213|
+-------+---+------+----------+--------+
only showing top 5 rows


- Prikazati 50 najpopularnijih filmova.

In [65]:
from pyspark.sql.functions import count, avg, round, max, col, row_number, array, when, explode

movie_ratings = ratings.join(movies, on="movie_id", how="inner")

popular_movies = movie_ratings.groupBy("movie_id", "title").agg(
    count("*").alias("views")
).orderBy("views", ascending=False)
popular_movies.show(50, truncate=False)

+--------+--------------------------------------------+-----+
|movie_id|title                                       |views|
+--------+--------------------------------------------+-----+
|50      |Star Wars (1977)                            |583  |
|258     |Contact (1997)                              |509  |
|100     |Fargo (1996)                                |508  |
|181     |Return of the Jedi (1983)                   |507  |
|294     |Liar Liar (1997)                            |485  |
|286     |English Patient, The (1996)                 |481  |
|288     |Scream (1996)                               |478  |
|1       |Toy Story (1995)                            |452  |
|300     |Air Force One (1997)                        |431  |
|121     |Independence Day (ID4) (1996)               |429  |
|174     |Raiders of the Lost Ark (1981)              |420  |
|127     |Godfather, The (1972)                       |413  |
|56      |Pulp Fiction (1994)                         |394  |
|7      

- Za svaki film izračunati broj pregleda i prosečnu ocenu i prikazati nazive 10 filmova sa
najviše i 10 filmova sa najmanje pregleda.

In [66]:
movie_stats = movie_ratings.groupBy("movie_id", "title").agg(
    count("*").alias("views"),
    round(avg("rating"), 2).alias("avg_rating")
)

print("Top 10 movies with the most views:")
movie_stats.orderBy("views", ascending=False).show(10, truncate=False)

print("Top 10 movies with the least views:")
movie_stats.orderBy("views", ascending=True).show(10, truncate=False)

Top 10 movies with the most views:
+--------+-----------------------------+-----+----------+
|movie_id|title                        |views|avg_rating|
+--------+-----------------------------+-----+----------+
|50      |Star Wars (1977)             |583  |4.36      |
|258     |Contact (1997)               |509  |3.8       |
|100     |Fargo (1996)                 |508  |4.16      |
|181     |Return of the Jedi (1983)    |507  |4.01      |
|294     |Liar Liar (1997)             |485  |3.16      |
|286     |English Patient, The (1996)  |481  |3.66      |
|288     |Scream (1996)                |478  |3.44      |
|1       |Toy Story (1995)             |452  |3.88      |
|300     |Air Force One (1997)         |431  |3.63      |
|121     |Independence Day (ID4) (1996)|429  |3.44      |
+--------+-----------------------------+-----+----------+
only showing top 10 rows
Top 10 movies with the least views:
+--------+-----------------------------------------------+-----+----------+
|movie_id|title 

- Izračunati “skor” za svaki film na osnovu sledeće formule:
(broj_pregleda*prosečna_ocena)/(max(broj_pregleda)*max(prosečna ocena)). Prikazati top
5 filmova na osnovu izračunatog „skora“.

In [67]:
max_views = movie_stats.agg(max("views")).collect()[0][0]
max_avg_rating = movie_stats.agg(max("avg_rating")).collect()[0][0]

movie_scores = movie_stats.withColumn(
    "score",
    round((col("views") * col("avg_rating")) / (max_views * max_avg_rating), 4)
).orderBy("score", ascending=False).show(5, truncate=False)

+--------+------------------------------+-----+----------+------+
|movie_id|title                         |views|avg_rating|score |
+--------+------------------------------+-----+----------+------+
|50      |Star Wars (1977)              |583  |4.36      |0.872 |
|100     |Fargo (1996)                  |508  |4.16      |0.725 |
|181     |Return of the Jedi (1983)     |507  |4.01      |0.6975|
|258     |Contact (1997)                |509  |3.8       |0.6635|
|174     |Raiders of the Lost Ark (1981)|420  |4.25      |0.6123|
+--------+------------------------------+-----+----------+------+
only showing top 5 rows


- Prikazati 5 najpopularnijih filmova po polu.

In [68]:
from pyspark.sql.window import Window

movie_ratings_users = movie_ratings.join(users, on="user_id", how="inner")
gender_popularity = movie_ratings_users.groupBy("gender", "title").agg(
    count("*").alias("views")
)

window_spec_gender = Window.partitionBy("gender").orderBy(col("views").desc())
ranked_movies_gender = gender_popularity.withColumn(
    "rank",
    row_number().over(window_spec_gender)
)

top_5_by_gender = ranked_movies_gender.filter(col("rank") <= 5)
top_5_by_gender.orderBy("gender", "rank").show(truncate=False)

+------+---------------------------+-----+----+
|gender|title                      |views|rank|
+------+---------------------------+-----+----+
|F     |English Patient, The (1996)|152  |1   |
|F     |Star Wars (1977)           |151  |2   |
|F     |Scream (1996)              |143  |3   |
|F     |Liar Liar (1997)           |141  |4   |
|F     |Contact (1997)             |137  |5   |
|M     |Star Wars (1977)           |432  |1   |
|M     |Return of the Jedi (1983)  |383  |2   |
|M     |Fargo (1996)               |383  |3   |
|M     |Contact (1997)             |372  |4   |
|M     |Liar Liar (1997)           |344  |5   |
+------+---------------------------+-----+----+



- Prikazati 3 najpopularnija filma po polu i zanimanju korisnika.

In [69]:
gender_occupation_popularity = movie_ratings_users.groupBy("gender", "occupation", "title").agg(
    count("*").alias("views")
)

window_spec_gender_occupation = Window.partitionBy("gender", "occupation").orderBy(col("views").desc())
ranked_movies_gender_occupation = gender_occupation_popularity.withColumn(
    "rank",
    row_number().over(window_spec_gender_occupation)
)

top_3_by_gender_and_occupation = ranked_movies_gender_occupation.filter(col("rank") <= 3)
top_3_by_gender_and_occupation.orderBy("gender", "occupation", "rank").show(truncate=False)

+------+-------------+------------------------------------+-----+----+
|gender|occupation   |title                               |views|rank|
+------+-------------+------------------------------------+-----+----+
|F     |administrator|English Patient, The (1996)         |21   |1   |
|F     |administrator|Star Wars (1977)                    |21   |2   |
|F     |administrator|Jerry Maguire (1996)                |19   |3   |
|F     |artist       |Contact (1997)                      |10   |1   |
|F     |artist       |Godfather, The (1972)               |8    |2   |
|F     |artist       |Chasing Amy (1997)                  |8    |3   |
|F     |educator     |English Patient, The (1996)         |23   |1   |
|F     |educator     |Contact (1997)                      |15   |2   |
|F     |educator     |Full Monty, The (1997)              |14   |3   |
|F     |engineer     |Evita (1996)                        |2    |1   |
|F     |engineer     |English Patient, The (1996)         |2    |2   |
|F    

- Prikazati 3 najpopularnija filma u svakom žanru.

In [70]:
movies_raw = spark.read.csv(
    f"{dataset_path}/u.item",
    sep="|",
    inferSchema=True,
    header=False

)
movies_full = movies_raw.toDF(
    "movie_id", "title", "release_date", "video_release_date",
    "imdb_url", "unknown", "Action", "Adventure", "Animation",
    "Children", "Comedy", "Crime", "Documentary", "Drama",
    "Fantasy", "FilmNoir", "Horror", "Musical", "Mystery",
    "Romance", "SciFi", "Thriller", "War", "Western"
)

movies_genres = movies_full.select(
    "movie_id",
    "title",
    array(
        when(col("Action") == 1, "Action"),
        when(col("Adventure") == 1, "Adventure"),
        when(col("Animation") == 1, "Animation"),
        when(col("Children") == 1, "Children"),
        when(col("Comedy") == 1, "Comedy"),
        when(col("Crime") == 1, "Crime"),
        when(col("Documentary") == 1, "Documentary"),
        when(col("Drama") == 1, "Drama"),
        when(col("Fantasy") == 1, "Fantasy"),
        when(col("FilmNoir") == 1, "Film-Noir"),
        when(col("Horror") == 1, "Horror"),
        when(col("Musical") == 1, "Musical"),
        when(col("Mystery") == 1, "Mystery"),
        when(col("Romance") == 1, "Romance"),
        when(col("SciFi") == 1, "Sci-Fi"),
        when(col("Thriller") == 1, "Thriller"),
        when(col("War") == 1, "War"),
        when(col("Western") == 1, "Western")
    ).alias("genres")
)

movies_by_genre = movies_genres.withColumn(
    "genre",
    explode(col("genres"))
).filter(col("genre").isNotNull())

movie_with_ratings = ratings.join(movies_by_genre, on="movie_id", how="inner")

genre_popularity = movie_with_ratings.groupBy("genre", "title").agg(count("*").alias("views"))

window_spec_genre = Window.partitionBy("genre").orderBy(col("views").desc())

ranked_movies_genre = genre_popularity.withColumn(
    "rank",
    row_number().over(window_spec_genre)
)

top_3_genres = ranked_movies_genre.filter(col("rank") <= 3)

top_3_genres.orderBy("genre", "rank").show(truncate=False)

+-----------+--------------------------------------------+-----+----+
|genre      |title                                       |views|rank|
+-----------+--------------------------------------------+-----+----+
|Action     |Star Wars (1977)                            |583  |1   |
|Action     |Return of the Jedi (1983)                   |507  |2   |
|Action     |Air Force One (1997)                        |431  |3   |
|Adventure  |Star Wars (1977)                            |583  |1   |
|Adventure  |Return of the Jedi (1983)                   |507  |2   |
|Adventure  |Raiders of the Lost Ark (1981)              |420  |3   |
|Animation  |Toy Story (1995)                            |452  |1   |
|Animation  |Lion King, The (1994)                       |220  |2   |
|Animation  |Aladdin (1992)                              |219  |3   |
|Children   |Toy Story (1995)                            |452  |1   |
|Children   |Willy Wonka and the Chocolate Factory (1971)|326  |2   |
|Children   |E.T. th

- Kreirati atribut kategorički starosna_grupa na osnovu atributa age ) i prikazati 5
najpopularnijih filmova za svaku grupu.

In [71]:
data = movie_ratings_users.withColumn(
    "starosna_grupa",
    when(col("age") <= 17, "teen")
    .when((col("age") >= 18) & (col("age") <= 25), "young-adult")
    .when((col("age") >= 26) & (col("age") <= 50), "adult")
    .otherwise("senior")
)

grouped_starosna_grupa = data.groupBy("starosna_grupa", "title").agg(count("*").alias("views"))

window_spec_starosna_grupa = Window.partitionBy("starosna_grupa").orderBy(col("views").desc())
ranked_starosna_grupa = grouped_starosna_grupa.withColumn(
    "rank",
    row_number().over(window_spec_starosna_grupa)
)

top_5_age = ranked_starosna_grupa.filter(col("rank") <= 5)
top_5_age.orderBy("starosna_grupa", "rank").show(truncate=False)

+--------------+---------------------------+-----+----+
|starosna_grupa|title                      |views|rank|
+--------------+---------------------------+-----+----+
|adult         |Star Wars (1977)           |362  |1   |
|adult         |Fargo (1996)               |314  |2   |
|adult         |Return of the Jedi (1983)  |307  |3   |
|adult         |Contact (1997)             |303  |4   |
|adult         |English Patient, The (1996)|300  |5   |
|senior        |English Patient, The (1996)|80   |1   |
|senior        |Fargo (1996)               |60   |2   |
|senior        |Air Force One (1997)       |54   |3   |
|senior        |Full Monty, The (1997)     |50   |4   |
|senior        |Godfather, The (1972)      |48   |5   |
|teen          |Scream (1996)              |28   |1   |
|teen          |Liar Liar (1997)           |21   |2   |
|teen          |Contact (1997)             |20   |3   |
|teen          |Star Wars (1977)           |19   |4   |
|teen          |Return of the Jedi (1983)  |18  

- Prikazati ukupnu prosečnu ocenu svih filmova.

In [72]:
overall_avg_ratings = ratings.agg(round(avg("rating"), 2).alias("global_avg_rating"))
overall_avg_ratings.show()

+-----------------+
|global_avg_rating|
+-----------------+
|             3.53|
+-----------------+



2. Sistem preporuke baziran na sadržaju

- Koristeći attribute age, gender, occupation, genre i movie year (ovaj atribut je potrebno
ekstrahovati iz naziva filma) i izlaznu varijablu ratings kreirati regresioni model korišćenjem
Random forest i Linearne regresije. Za svaki algoritam optimizujte hiper-parametre
korišćenjem Cross-validacije. Za trening skup koristite u1.base a kao test skup u1.test.

In [77]:
from pyspark.sql.functions import regexp_extract, expr

movies_year = movies_full.withColumn(
    "year_str",
    regexp_extract(col("title"), r"\((\d{4})\)", 1)
).withColumn(
    "year",
    when(col("year_str") != "", col("year_str").cast("int")).otherwise(None)
)

movies_features = movies_year.withColumn(
    "genres",
    expr("""
        filter(array(
            case when Action = 1 then 'Action' end,
            case when Adventure = 1 then 'Adventure' end,
            case when Animation = 1 then 'Animation' end,
            case when Children = 1 then 'Children' end,
            case when Comedy = 1 then 'Comedy' end,
            case when Crime = 1 then 'Crime' end,
            case when Documentary = 1 then 'Documentary' end,
            case when Drama = 1 then 'Drama' end,
            case when Fantasy = 1 then 'Fantasy' end,
            case when FilmNoir = 1 then 'Film-Noir' end,
            case when Horror = 1 then 'Horror' end,
            case when Musical = 1 then 'Musical' end,
            case when Mystery = 1 then 'Mystery' end,
            case when Romance = 1 then 'Romance' end,
            case when SciFi = 1 then 'Sci-Fi' end,
            case when Thriller = 1 then 'Thriller' end,
            case when War = 1 then 'War' end,
            case when Western = 1 then 'Western' end
        ), x -> x is not null)
    """)
)

model_data = ratings.join(users, on="user_id") \
    .join(movies_features.select("movie_id", "genres", "year"), on="movie_id") \
    .select("age", "gender", "occupation", "genres", "year", "rating")

model_data.show(5, truncate=False)

+---+------+----------+-------------------------------------+----+------+
|age|gender|occupation|genres                               |year|rating|
+---+------+----------+-------------------------------------+----+------+
|49 |M     |writer    |[Comedy]                             |1996|3     |
|39 |F     |executive |[Crime, Film-Noir, Mystery, Thriller]|1997|3     |
|25 |M     |writer    |[Children, Comedy]                   |1994|1     |
|28 |M     |technician|[Drama, Romance, War, Western]       |1994|2     |
|47 |M     |educator  |[Crime, Drama]                       |1997|1     |
+---+------+----------+-------------------------------------+----+------+
only showing top 5 rows


In [78]:
train_ratings = spark.read.csv(
    f"{dataset_path}/u1.base",
    sep="\t",
    inferSchema=True,
    header=False
).toDF("user_id", "movie_id", "rating", "timestamp")

test_ratings = spark.read.csv(
    f"{dataset_path}/u1.test",
    sep="\t",
    inferSchema=True,
    header=False
).toDF("user_id", "movie_id", "rating", "timestamp")

train_data = train_ratings.join(users, on="user_id") \
    .join(movies_features.select("movie_id", "genres", "year"), on="movie_id") \
    .select("age", "gender", "occupation", "genres", "year", "rating") \
    .filter(col("year").isNotNull())

train_data.show(5, truncate=False)

test_data = test_ratings.join(users, on="user_id") \
    .join(movies_features.select("movie_id", "genres", "year"), on="movie_id") \
    .select("age", "gender", "occupation", "genres", "year", "rating") \
    .filter(col("year").isNotNull())

test_data.show(5, truncate=False)

+---+------+----------+-----------------------------+----+------+
|age|gender|occupation|genres                       |year|rating|
+---+------+----------+-----------------------------+----+------+
|24 |M     |technician|[Animation, Children, Comedy]|1995|5     |
|24 |M     |technician|[Action, Adventure, Thriller]|1995|3     |
|24 |M     |technician|[Thriller]                   |1995|4     |
|24 |M     |technician|[Action, Comedy, Drama]      |1995|3     |
|24 |M     |technician|[Crime, Drama, Thriller]     |1995|3     |
+---+------+----------+-----------------------------+----+------+
only showing top 5 rows
+---+------+----------+-----------------------------------------+----+------+
|age|gender|occupation|genres                                   |year|rating|
+---+------+----------+-----------------------------------------+----+------+
|24 |M     |technician|[Drama]                                  |1995|5     |
|24 |M     |technician|[Drama, War]                             |1995|

In [79]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, CountVectorizer
from pyspark.ml import Pipeline

gender_indexer = StringIndexer(inputCol="gender", outputCol="gender_index", handleInvalid="keep")
occupation_indexer = StringIndexer(inputCol="occupation", outputCol="occupation_index", handleInvalid="keep")

cv = CountVectorizer(inputCol="genres", outputCol="genre_vec")

encoder = OneHotEncoder(
    inputCols=["gender_index", "occupation_index"],
    outputCols=["gender_vec", "occupation_vec"]
)

num_assembler = VectorAssembler(
    inputCols=["age", "year"],
    outputCol="num_features"
)

scaler = StandardScaler(
    inputCol="num_features",
    outputCol="scaled_num_features",
    withMean=True,
    withStd=True
)

assembler = VectorAssembler(
    inputCols=["scaled_num_features", "gender_vec", "occupation_vec", "genre_vec"],
    outputCol="features"
)

rf_assembler = VectorAssembler(
    inputCols=["age", "year", "gender_vec", "occupation_vec", "genre_vec"],
    outputCol="features"
)

In [80]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

lr = LinearRegression(
    featuresCol="features",
    labelCol="rating"
)

lr_pipeline = Pipeline(stages=[
    gender_indexer,
    occupation_indexer,
    encoder,
    cv,
    num_assembler,
    scaler,
    assembler,
    lr
])

lr_paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

evaluator = RegressionEvaluator(
    labelCol="rating",
    predictionCol="prediction",
    metricName="rmse"
)

lr_cv = CrossValidator(
    estimator=lr_pipeline,
    estimatorParamMaps=lr_paramGrid,
    evaluator=evaluator,
    numFolds=3
)

lr_cv_model = lr_cv.fit(train_data)

lr_predictions = lr_cv_model.transform(test_data)

lr_rmse = evaluator.evaluate(lr_predictions)
print("Linear Regression RMSE:", lr_rmse)

best_lr = lr_cv_model.bestModel.stages[-1]
print("Best regParam:", best_lr._java_obj.getRegParam())
print("Best elasticNetParam:", best_lr._java_obj.getElasticNetParam())

Linear Regression RMSE: 1.1042903160648714
Best regParam: 0.01
Best elasticNetParam: 0.0


In [81]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="rating"
)

rf_pipeline = Pipeline(stages=[
    gender_indexer,
    occupation_indexer,
    cv,
    encoder,
    rf_assembler,
    rf
])

rf_paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 50]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_paramGrid,
    evaluator=evaluator,
    numFolds=3
)

rf_cv_model = rf_cv.fit(train_data)

rf_predictions = rf_cv_model.transform(test_data)

rf_rmse = evaluator.evaluate(rf_predictions)
print("Random Forest RMSE:", rf_rmse)

best_rf = rf_cv_model.bestModel.stages[-1]
print("Best numTrees:", best_rf.getNumTrees)
print("Best maxDepth:", best_rf.getOrDefault("maxDepth"))

26/04/28 03:37:39 WARN DAGScheduler: Broadcasting large task binary with size 1374.8 KiB
26/04/28 03:37:39 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/28 03:37:44 WARN DAGScheduler: Broadcasting large task binary with size 1137.8 KiB
26/04/28 03:37:45 WARN DAGScheduler: Broadcasting large task binary with size 1970.5 KiB
26/04/28 03:37:45 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/04/28 03:37:46 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/28 03:37:50 WARN DAGScheduler: Broadcasting large task binary with size 1373.5 KiB
26/04/28 03:37:51 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB
26/04/28 03:37:56 WARN DAGScheduler: Broadcasting large task binary with size 1134.6 KiB
26/04/28 03:37:57 WARN DAGScheduler: Broadcasting large task binary with size 1970.5 KiB
26/04/28 03:37:57 WARN DAGScheduler: Broadcasting large task binary with size 3.2 MiB
26/04/28 03:37:58 WARN DAGScheduler:

Random Forest RMSE: 1.0659523389813865
Best numTrees: 50
Best maxDepth: 10


In [82]:
best_regression_model = rf_cv_model if rf_rmse <= lr_rmse else lr_cv_model
best_regression_name = "Random Forest" if rf_rmse <= lr_rmse else "Linear Regression"
best_regression_rmse = min(rf_rmse, lr_rmse)
print(f"Best regression model: {best_regression_name} (RMSE={best_regression_rmse:.4f})")

Best regression model: Random Forest (RMSE=1.0660)


- Napravite proceduru za preporuku na sledeći način:
o Ulaz: korisnik (age, gender, occupation), 50 najpopularnijih filmova određenih u
prethodnom zadatku (genre, year).
o Za datog korisnika predvideti ocene na osnovu modela sa najboljom performansom
o Izlaz: Rangirani filmovi

In [83]:
top_50_movies = popular_movies.limit(50) \
    .join(movies_features.select("movie_id", "genres", "year"), on="movie_id") \
    .select("title", "genres", "year")

top_50_movies.show(5, truncate=False)

+-------------------------+-----------------------------+----+
|title                    |genres                       |year|
+-------------------------+-----------------------------+----+
|Toy Story (1995)         |[Animation, Children, Comedy]|1995|
|Twelve Monkeys (1995)    |[Drama, Sci-Fi]              |1995|
|Dead Man Walking (1995)  |[Drama]                      |1995|
|Mr. Holland's Opus (1995)|[Drama]                      |1995|
|Braveheart (1995)        |[Action, Drama, War]         |1995|
+-------------------------+-----------------------------+----+
only showing top 5 rows


In [84]:
def recommend_movies(age, gender, occupation, top_movies_df):
    from pyspark.sql.functions import lit

    input_df = top_movies_df \
        .withColumn("age", lit(age)) \
        .withColumn("gender", lit(gender)) \
        .withColumn("occupation", lit(occupation))

    predictions = best_regression_model.transform(input_df)

    ranked = predictions.select(
        "title",
        "genres",
        "year",
        "prediction"
    ).orderBy("prediction", ascending=False)

    return ranked

In [85]:
result = recommend_movies(
    age=25,
    gender="M",
    occupation="student",
    top_movies_df=top_50_movies
)

result.show(10, truncate=False)

+--------------------------------+------------------------------------------------+----+------------------+
|title                           |genres                                          |year|prediction        |
+--------------------------------+------------------------------------------------+----+------------------+
|Star Wars (1977)                |[Action, Adventure, Romance, Sci-Fi, War]       |1977|4.265071791250278 |
|Empire Strikes Back, The (1980) |[Action, Adventure, Drama, Romance, Sci-Fi, War]|1980|4.21075088058938  |
|Godfather, The (1972)           |[Action, Crime, Drama]                          |1972|4.20393561267384  |
|Silence of the Lambs, The (1991)|[Drama, Thriller]                               |1991|4.037911214670668 |
|Amadeus (1984)                  |[Drama, Mystery]                                |1984|4.037220899908466 |
|Return of the Jedi (1983)       |[Action, Adventure, Romance, Sci-Fi, War]       |1983|4.024278252584087 |
|Alien (1979)               

3. Sistem preporuke baziran na kolaborativnom filtriranju

- Koristeći skup podataka u1.base cross validacijom optimizovati hiperparametre ALS
modela. Performanse najboljeg modela proceniti na test skupu podataka (u1.test).
Diskutovati da li postoji pre-treniranje ili pod-treniranje modela.

In [86]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

train_ratings = spark.read.csv(
    f"{dataset_path}/u1.base",
    sep="\t",
    inferSchema=True
).toDF("user_id", "movie_id", "rating", "timestamp")

test_ratings = spark.read.csv(
    f"{dataset_path}/u1.test",
    sep="\t",
    inferSchema=True
).toDF("user_id", "movie_id", "rating", "timestamp")

als = ALS(
    userCol="user_id",
    itemCol="movie_id",
    ratingCol="rating",
    coldStartStrategy="drop",
    nonnegative=True
)

als_paramGrid = ParamGridBuilder() \
    .addGrid(als.rank, [5, 10, 20]) \
    .addGrid(als.regParam, [0.1, 0.2, 0.5]) \
    .addGrid(als.maxIter, [5, 10]) \
    .build()

evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

als_cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=als_paramGrid,
    evaluator=evaluator,
    numFolds=3
)

als_cv_model = als_cv.fit(train_ratings)

train_predictions = als_cv_model.transform(train_ratings)
train_rmse = evaluator.evaluate(train_predictions)
print("ALS Train RMSE:", train_rmse)

test_predictions = als_cv_model.transform(test_ratings)
test_rmse = evaluator.evaluate(test_predictions)
print("ALS Test RMSE:", test_rmse)

als_best_model = als_cv_model.bestModel
print("Best rank:", als_best_model.rank)
print("Best regParam:", als_best_model._java_obj.parent().getRegParam())
print("Best maxIter:", als_best_model._java_obj.parent().getMaxIter())

ALS Train RMSE: 0.8165580692946633
ALS Test RMSE: 0.9312257269134032
Best rank: 5
Best regParam: 0.1
Best maxIter: 10


In [87]:
rmse_gap = test_rmse - train_rmse
print(f"RMSE gap (test - train): {rmse_gap:.4f}")

if rmse_gap > 0.15:
    print("Zakljucak: model pokazuje znake overfitting-a.")
elif train_rmse > 1.2 and test_rmse > 1.2:
    print("Zakljucak: model je verovatno underfitted (visok train i test RMSE).")
else:
    print("Zakljucak: nema jakih znakova overfitting-a; generalizacija je solidna.")

RMSE gap (test - train): 0.1147
Zakljucak: nema jakih znakova overfitting-a; generalizacija je solidna.


- Korišćenjem u2.base identifikujte korisnike koji su dali manje od 5 ocena ili se ne pojavljuju
u u1.base i napravite dve promenljive: users_with_grades (korisnici iz u2.base koji se
pojavljuju u u1.base i imaju minimum 5 ocena) i users_without_grades (korisnici iz u2.base
koji se ne pojavljuju u u1.base ili imaju manje od 5 ocena)). Takođe, odvojiti filmove u dve
različite promenljive: movies_u2_only – filmovi koji se ne nalaze u u1.base a nalaze se u
u2_base i movies_u_12 – filmovi koji se nalaze u obe taabele.

In [88]:
from pyspark.sql.functions import count

u2_ratings = spark.read.csv(
    f"{dataset_path}/u2.base",
    sep="\t",
    inferSchema=True
).toDF("user_id", "movie_id", "rating", "timestamp")

u2_user_counts = u2_ratings.groupBy("user_id").agg(
    count("*").alias("num_ratings")
)

u1_users = train_ratings.select("user_id").distinct()

users_with_grades = u2_user_counts.join(
    u1_users,
    on="user_id",
    how="inner"
).filter("num_ratings >= 5")

users_without_grades = u2_user_counts.join(
    u1_users,
    on="user_id",
    how="left_anti"
).union(
    u2_user_counts.join(
        u1_users,
        on="user_id",
        how="inner"
    ).filter("num_ratings < 5")
)

u1_movies = train_ratings.select("movie_id").distinct()
u2_movies = u2_ratings.select("movie_id").distinct()

movies_u2_only = u2_movies.join(
    u1_movies,
    on="movie_id",
    how="left_anti"
)

movies_u_12 = u2_movies.join(
    u1_movies,
    on="movie_id",
    how="inner"
)

print("Users with grades:", users_with_grades.count())
print("Users without grades:", users_without_grades.count())

print("Movies only in u2:", movies_u2_only.count())
print("Movies in both:", movies_u_12.count())

Users with grades: 943
Users without grades: 0
Movies only in u2: 32
Movies in both: 1616


- Napraviti proceduru koja će deliti korisnike iz u2.base na osnovu prethodnog opisa i koja će
za users_with_grades previđati na osnovu ALS-a, a za users_without_grades predviđati na
osnovu regresionog modela iz prethodne stavke. Na izlazu procedure treba da postoje
predikcije za sve korisnike.

In [89]:
users_with_profiles = users_with_grades.select("user_id").join(
    users.select("user_id", "age", "gender", "occupation"),
    on="user_id",
    how="inner"
 )

users_without_profiles = users_without_grades.select("user_id").join(
    users.select("user_id", "age", "gender", "occupation"),
    on="user_id",
    how="inner"
 )

print("users_with_profiles:", users_with_profiles.count())
print("users_without_profiles:", users_without_profiles.count())

users_with_profiles: 943
users_without_profiles: 0


In [90]:
def build_hybrid_predictions():
    empty_schema = "user_id INT, movie_id INT, prediction DOUBLE"

    if users_with_profiles.rdd.isEmpty():
        als_predictions = spark.createDataFrame([], empty_schema)
    else:
        als_input = users_with_profiles.select("user_id").distinct().crossJoin(
            movies_u_12.select("movie_id").distinct()
        )
        als_predictions = als_best_model.transform(als_input).select(
            "user_id", "movie_id", "prediction"
        )

    if users_without_profiles.rdd.isEmpty():
        reg_predictions = spark.createDataFrame([], empty_schema)
    else:
        reg_candidate_movies = (
            u2_movies.join(
                movies_features.select("movie_id", "genres", "year"),
                on="movie_id",
                how="inner"
            )
            .filter(col("year").isNotNull())
            .select("movie_id", "genres", "year")
            .distinct()
        )

        reg_input = users_without_profiles.crossJoin(reg_candidate_movies)
        reg_predictions = best_regression_model.transform(reg_input).select(
            "user_id", "movie_id", "prediction"
        )

    return als_predictions.unionByName(reg_predictions)

In [91]:
all_hybrid_predictions = build_hybrid_predictions()

seen = u2_ratings.select("user_id", "movie_id")

all_hybrid_predictions = all_hybrid_predictions.join(
    seen,
    on=["user_id", "movie_id"],
    how="left_anti"
)

print("Total hybrid predictions:", all_hybrid_predictions.count())
print(
    "Distinct users covered:",
    all_hybrid_predictions.select("user_id").distinct().count()
 )

all_hybrid_predictions.show(10, truncate=False)

Total hybrid predictions: 1443920
Distinct users covered: 943
+-------+--------+------------------+
|user_id|movie_id|prediction        |
+-------+--------+------------------+
|148    |148     |3.2766666412353516|
|148    |463     |2.9813854694366455|
|148    |471     |3.4880118370056152|
|148    |496     |3.7790937423706055|
|148    |833     |2.6599154472351074|
|148    |1088    |1.7598435878753662|
|148    |1238    |2.802563190460205 |
|148    |1342    |2.6078925132751465|
|148    |1591    |1.6947320699691772|
|148    |1645    |4.224458694458008 |
+-------+--------+------------------+
only showing top 10 rows


- Primenite proceduru iz prethodnog opisa i prikažite preporuke za korisnika koji je imao
najviše ocena u u2.base.

In [92]:
u2_base = spark.read.csv(
    f"{dataset_path}/u2.base",
    sep="\t",
    inferSchema=True
).toDF("user_id", "movie_id", "rating", "timestamp")

u2_base.show(5)

+-------+--------+------+---------+
|user_id|movie_id|rating|timestamp|
+-------+--------+------+---------+
|      1|       3|     4|878542960|
|      1|       4|     3|876893119|
|      1|       5|     3|889751712|
|      1|       6|     5|887431973|
|      1|       7|     4|875071561|
+-------+--------+------+---------+
only showing top 5 rows


In [93]:
from pyspark.sql.functions import count, desc

top_user = u2_base.groupBy("user_id") \
    .agg(count("*").alias("num_ratings")) \
    .orderBy(desc("num_ratings")) \
    .first()

top_user_id = top_user["user_id"]

print("User with most ratings:", top_user_id)
print("Number of ratings:", top_user["num_ratings"])

User with most ratings: 655
Number of ratings: 669


In [94]:
top_user_info = users.filter(col("user_id") == top_user_id).select(
    "age", "gender", "occupation"
).first()

user_age = top_user_info["age"]
user_gender = top_user_info["gender"]
user_occupation = top_user_info["occupation"]

print(user_age, user_gender, user_occupation)

50 F healthcare


In [95]:
recommendations = all_hybrid_predictions.filter(col("user_id") == top_user_id)
print("Predictions for top user:", recommendations.count())

Predictions for top user: 947


In [96]:
final_recommendations = recommendations.join(
    movies.select("movie_id", "title"),
    on="movie_id",
    how="left"
).select(
    "movie_id",
    "title",
    "prediction"
).orderBy("prediction", ascending=False)

final_recommendations.show(10, truncate=False)

+--------+-------------------------------------------+------------------+
|movie_id|title                                      |prediction        |
+--------+-------------------------------------------+------------------+
|1449    |Pather Panchali (1955)                     |3.896711826324463 |
|851     |Two or Three Things I Know About Her (1966)|3.805694103240967 |
|1512    |World of Apu, The (Apur Sansar) (1959)     |3.6489853858947754|
|1467    |Saint of Fort Washington, The (1993)       |3.632228136062622 |
|1558    |Aparajito (1956)                           |3.578623056411743 |
|1122    |They Made Me a Criminal (1939)             |3.542048454284668 |
|119     |Maya Lin: A Strong Clear Vision (1994)     |3.5387630462646484|
|361     |Incognito (1997)                           |3.500839948654175 |
|169     |Wrong Trousers, The (1993)                 |3.483241081237793 |
|745     |Ruling Class, The (1972)                   |3.4810614585876465|
+--------+----------------------------